In [ ]:
!pip install transformers datasets seqeval torch scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=d9a5e31330c16ab79ab331e4344336d4508ad98187a04658e4a54c1d43075de9
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


# Loading and Parsing BIO Data

In [ ]:
def read_bio_file(filepath):
    sentences = []
    labels = []

    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()

    blocks = content.strip().split("\n\n")

    for block in blocks:
        block = block.strip()
        if not block:
            continue

        words = []
        tags = []

        for line in block.split("\n"):
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) == 2:
                words.append(parts[0])
                tags.append(parts[1])

        if words:
            sentences.append(words)
            labels.append(tags)

    return sentences, labels

# Load all 3 splits
train_sentences, train_labels = read_bio_file("/content/climate_train.txt")
dev_sentences, dev_labels = read_bio_file("/content/climate_dev.txt")
test_sentences, test_labels = read_bio_file("/content/climate_test.txt")

print(f"Train: {len(train_sentences)} sentences")
print(f"Dev:   {len(dev_sentences)} sentences")
print(f"Test:  {len(test_sentences)} sentences")

# Preview first sentence
print(f"\nFirst training sentence:")
for word, tag in zip(train_sentences[0], train_labels[0]):
    print(f"  {word:20s} {tag}")

Train: 210 sentences
Dev:   26 sentences
Test:  27 sentences

First training sentence:
  Abstract.            O
  Glacier              B-Env_Event
  melt                 I-Env_Event
  is                   O
  an                   O
  important            O
  source               O
  of                   O
  water                O
  for                  O
  high                 O
  Andean               O
  rivers               O
  in                   O
  central              O
  Chile,               O
  especially           O
  in                   O
  dry                  O
  years,               O
  when                 O
  it                   O
  can                  O
  be                   O
  an                   O
  important            O
  contributor          O
  to                   O
  flows                O
  during               O
  late                 O
  summer               O
  and                  O
  autumn.              O
  However,             O
  few             

# Defining Label Mappings

In [ ]:
# Get all unique labels from training data
all_labels = set()
for label_seq in train_labels + dev_labels + test_labels:
    for label in label_seq:
        all_labels.add(label)

# Sort and create mappings
label_list = sorted(list(all_labels))
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

print(f"Total unique labels: {len(label_list)}")
print("\nLabel mappings:")
for label, idx in label2id.items():
    print(f"  {idx}: {label}")

Total unique labels: 33

Label mappings:
  0: B-CLIMATE_DRIVER
  1: B-CLIMATE_VARIABLE
  2: B-Climate_Driver
  3: B-Climate_Variable
  4: B-ECOSYSTEM
  5: B-ENVIRONMENTAL_EVENT
  6: B-Ecosystem
  7: B-Env_Event
  8: B-GEOGRAPHIC_LOCATION
  9: B-Geo_Location
  10: B-HUMAN_ACTIVITY
  11: B-Human_Activity
  12: B-POLICY
  13: B-Policy
  14: B-SPECIES
  15: B-Species
  16: I-CLIMATE_DRIVER
  17: I-CLIMATE_VARIABLE
  18: I-Climate_Driver
  19: I-Climate_Variable
  20: I-ECOSYSTEM
  21: I-ENVIRONMENTAL_EVENT
  22: I-Ecosystem
  23: I-Env_Event
  24: I-GEOGRAPHIC_LOCATION
  25: I-Geo_Location
  26: I-HUMAN_ACTIVITY
  27: I-Human_Activity
  28: I-POLICY
  29: I-Policy
  30: I-SPECIES
  31: I-Species
  32: O


In [ ]:
# Standardize all labels to one consistent format
label_mapping = {
    # Climate Driver
    "B-CLIMATE_DRIVER": "B-Climate_Driver",
    "I-CLIMATE_DRIVER": "I-Climate_Driver",
    # Climate Variable
    "B-CLIMATE_VARIABLE": "B-Climate_Variable",
    "I-CLIMATE_VARIABLE": "I-Climate_Variable",
    # Env Event
    "B-ENVIRONMENTAL_EVENT": "B-Env_Event",
    "I-ENVIRONMENTAL_EVENT": "I-Env_Event",
    # Ecosystem
    "B-ECOSYSTEM": "B-Ecosystem",
    "I-ECOSYSTEM": "I-Ecosystem",
    # Geo Location
    "B-GEOGRAPHIC_LOCATION": "B-Geo_Location",
    "I-GEOGRAPHIC_LOCATION": "I-Geo_Location",
    # Human Activity
    "B-HUMAN_ACTIVITY": "B-Human_Activity",
    "I-HUMAN_ACTIVITY": "I-Human_Activity",
    # Policy
    "B-POLICY": "B-Policy",
    "I-POLICY": "I-Policy",
    # Species
    "B-SPECIES": "B-Species",
    "I-SPECIES": "I-Species",
}

def standardize_labels(labels_list):
    standardized = []
    for label_seq in labels_list:
        new_seq = [label_mapping.get(label, label) for label in label_seq]
        standardized.append(new_seq)
    return standardized

# Apply to all splits
train_labels = standardize_labels(train_labels)
dev_labels = standardize_labels(dev_labels)
test_labels = standardize_labels(test_labels)

# Verify fix
all_labels = set()
for label_seq in train_labels + dev_labels + test_labels:
    for label in label_seq:
        all_labels.add(label)

label_list = sorted(list(all_labels))
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

print(f"Total unique labels after fix: {len(label_list)}")
print("\nCleaned label mappings:")
for label, idx in label2id.items():
    print(f"  {idx}: {label}")

Total unique labels after fix: 17

Cleaned label mappings:
  0: B-Climate_Driver
  1: B-Climate_Variable
  2: B-Ecosystem
  3: B-Env_Event
  4: B-Geo_Location
  5: B-Human_Activity
  6: B-Policy
  7: B-Species
  8: I-Climate_Driver
  9: I-Climate_Variable
  10: I-Ecosystem
  11: I-Env_Event
  12: I-Geo_Location
  13: I-Human_Activity
  14: I-Policy
  15: I-Species
  16: O


# SciBERT tokenizer

In [ ]:
from transformers import AutoTokenizer

model_name = "allenai/scibert_scivocab_uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"Tokenizer loaded: {model_name}")
print(f"Vocabulary size: {tokenizer.vocab_size}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Tokenizer loaded: allenai/scibert_scivocab_uncased
Vocabulary size: 31090


# Tokenize and Align Labels

In [ ]:
import torch
from torch.utils.data import Dataset

def tokenize_and_align_labels(sentences, labels, tokenizer, max_length=128):
    tokenized_inputs = tokenizer(
        sentences,
        is_split_into_words=True,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

    aligned_labels = []
    for i, label_seq in enumerate(labels):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_id = None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)  # Special tokens ignored in loss
            elif word_id != previous_word_id:
                label_ids.append(label2id[label_seq[word_id]])
            else:
                label_ids.append(-100)  # Subword tokens ignored
            previous_word_id = word_id
        aligned_labels.append(label_ids)

    tokenized_inputs["labels"] = torch.tensor(aligned_labels)
    return tokenized_inputs


class NERDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return self.encodings["input_ids"].shape[0]

    def __getitem__(self, idx):
        return {key: val[idx] for key, val in self.encodings.items()}


# Tokenize all splits
print("Tokenizing training data...")
train_encodings = tokenize_and_align_labels(train_sentences, train_labels, tokenizer)
print("Tokenizing dev data...")
dev_encodings = tokenize_and_align_labels(dev_sentences, dev_labels, tokenizer)
print("Tokenizing test data...")
test_encodings = tokenize_and_align_labels(test_sentences, test_labels, tokenizer)

# Create datasets
train_dataset = NERDataset(train_encodings)
dev_dataset = NERDataset(dev_encodings)
test_dataset = NERDataset(test_encodings)

print(f"\nTrain dataset: {len(train_dataset)} samples")
print(f"Dev dataset:   {len(dev_dataset)} samples")
print(f"Test dataset:  {len(test_dataset)} samples")
print("\nTokenization complete!")

Tokenizing training data...
Tokenizing dev data...
Tokenizing test data...

Train dataset: 210 samples
Dev dataset:   26 samples
Test dataset:  27 samples

Tokenization complete!


# Load SciBERT model

In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

# Move to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"Model loaded: {model_name}")
print(f"Device: {device}")
print(f"Number of labels: {len(label_list)}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

pytorch_model.bin:   0%|          | 0.00/442M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

BertForTokenClassification LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored 

Model loaded: allenai/scibert_scivocab_uncased
Device: cuda
Number of labels: 17
Model parameters: 109,340,945


# TRAINING SETUP

In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# Training hyperparameters
EPOCHS = 5
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WARMUP_STEPS = 50

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Optimizer and scheduler
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=total_steps
)

print(f"Training configuration:")
print(f"  Epochs:        {EPOCHS}")
print(f"  Batch size:    {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Total steps:   {total_steps}")
print(f"  Train batches: {len(train_loader)}")
print(f"  Dev batches:   {len(dev_loader)}")

Training configuration:
  Epochs:        5
  Batch size:    16
  Learning rate: 2e-05
  Total steps:   70
  Train batches: 14
  Dev batches:   2


# Training

In [ ]:
from seqeval.metrics import f1_score, precision_score, recall_score
import numpy as np

def evaluate(model, data_loader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            predictions = torch.argmax(outputs.logits, dim=2)

            for pred_seq, label_seq in zip(predictions, labels):
                pred_list = []
                label_list = []
                for pred, label in zip(pred_seq, label_seq):
                    if label.item() != -100:
                        pred_list.append(id2label[pred.item()])
                        label_list.append(id2label[label.item()])
                all_preds.append(pred_list)
                all_labels.append(label_list)

    f1 = f1_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)
    return f1, precision, recall


# Training loop
print("Starting training...")
print("=" * 60)

best_dev_f1 = 0
best_epoch = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

    avg_loss = total_loss / len(train_loader)

    # Evaluate on dev set
    dev_f1, dev_precision, dev_recall = evaluate(model, dev_loader, device)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"  Train Loss:    {avg_loss:.4f}")
    print(f"  Dev F1:        {dev_f1:.4f}")
    print(f"  Dev Precision: {dev_precision:.4f}")
    print(f"  Dev Recall:    {dev_recall:.4f}")

    # Save best model
    if dev_f1 > best_dev_f1:
        best_dev_f1 = dev_f1
        best_epoch = epoch + 1
        torch.save(model.state_dict(), "best_scibert_ner.pt")
        print(f"  ✓ Best model saved!")
    print("-" * 40)

print(f"\nTraining complete!")
print(f"Best Dev F1: {best_dev_f1:.4f} at epoch {best_epoch}")

Starting training...
Epoch 1/5
  Train Loss:    2.8229
  Dev F1:        0.0000
  Dev Precision: 0.0000
  Dev Recall:    0.0000
----------------------------------------


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Epoch 2/5
  Train Loss:    1.4917
  Dev F1:        0.0000
  Dev Precision: 0.0000
  Dev Recall:    0.0000
----------------------------------------
Epoch 3/5
  Train Loss:    0.9910
  Dev F1:        0.0000
  Dev Precision: 0.0000
  Dev Recall:    0.0000
----------------------------------------
Epoch 4/5
  Train Loss:    0.7549
  Dev F1:        0.1864
  Dev Precision: 0.2303
  Dev Recall:    0.1565
  ✓ Best model saved!
----------------------------------------
Epoch 5/5
  Train Loss:    0.6705
  Dev F1:        0.2227
  Dev Precision: 0.2753
  Dev Recall:    0.1870
  ✓ Best model saved!
----------------------------------------

Training complete!
Best Dev F1: 0.2227 at epoch 5


# GENERATING SILVER LABELS

In [ ]:
import pandas as pd
import torch

# Load abstracts
df = pd.read_csv("/content/climate_abstracts.csv")
print(f"Total abstracts loaded: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
df.head(3)

Total abstracts loaded: 601
Columns: ['title', 'abstract', 'year', 'authors', 'venue', 'citation_count', 'fields']


,title,abstract,year,authors,venue,citation_count,fields
0,Episodic vs. Sea Level Rise Coastal Flooding S...,Sea level rise (SLR) and increased urbanisatio...,2025.0,"Sebastian Spadotto, Saverio Fracaros, A. Bezzi...",Water,0,"Environmental Science, Geography"
1,Future Coastal Population Growth and Exposure ...,Coastal zones are exposed to a range of coasta...,2015.0,"B. Neumann, A. Vafeidis, J. Zimmermann, R. Nic...",PLoS ONE,2301,"Medicine, Geography, Environmental Science, Ge..."
2,New elevation data triple estimates of global ...,Most estimates of global mean sea-level rise t...,2019.0,"S. Kulp, B. Strauss",Nature Communications,1037,"Environmental Science, Medicine, Environmental..."


# Split abstracts into sentences and generate silver labels

In [ ]:
import re

def split_into_sentences(text):
    sentences = re.split(r'(?<=[.!?])\s+', str(text))
    return [s.strip() for s in sentences if len(s.strip()) > 30]

def predict_silver_labels(sentences, model, tokenizer, device,
                           confidence_threshold=0.85):
    model.eval()
    silver_sentences = []
    silver_labels = []

    with torch.no_grad():
        for sent in sentences:
            words = sent.split()
            if not words:
                continue

            encoding = tokenizer(
                words,
                is_split_into_words=True,
                padding="max_length",
                truncation=True,
                max_length=128,
                return_tensors="pt"
            )

            input_ids = encoding["input_ids"].to(device)
            attention_mask = encoding["attention_mask"].to(device)

            outputs = model(input_ids=input_ids,
                          attention_mask=attention_mask)

            # Get probabilities
            probs = torch.softmax(outputs.logits, dim=2)
            predictions = torch.argmax(outputs.logits, dim=2)
            confidence = torch.max(probs, dim=2).values

            word_ids = encoding.word_ids(batch_index=0)

            pred_tags = []
            conf_scores = []
            seen_words = set()

            for idx, word_id in enumerate(word_ids):
                if word_id is None or word_id in seen_words:
                    continue
                seen_words.add(word_id)
                pred_tags.append(id2label[predictions[0][idx].item()])
                conf_scores.append(confidence[0][idx].item())

            # Only keep sentence if avg confidence is high
            avg_conf = sum(conf_scores) / len(conf_scores)
            if avg_conf >= confidence_threshold:
                silver_sentences.append(words[:len(pred_tags)])
                silver_labels.append(pred_tags)

    return silver_sentences, silver_labels

# Generate silver labels for all 601 abstracts
print("Generating silver labels for all abstracts...")
print("This may take 5-10 minutes...")

all_silver_sentences = []
all_silver_labels = []

for idx, row in df.iterrows():
    sentences = split_into_sentences(row["abstract"])
    s_sents, s_labels = predict_silver_labels(
        sentences, model, tokenizer, device
    )
    all_silver_sentences.extend(s_sents)
    all_silver_labels.extend(s_labels)

    if (idx + 1) % 100 == 0:
        print(f"  Processed {idx+1}/601 abstracts... "
              f"({len(all_silver_sentences)} silver sentences so far)")

print(f"\nSilver label generation complete!")
print(f"Total silver sentences: {len(all_silver_sentences)}")

Generating silver labels for all abstracts...
This may take 5-10 minutes...
  Processed 100/601 abstracts... (153 silver sentences so far)
  Processed 200/601 abstracts... (395 silver sentences so far)
  Processed 300/601 abstracts... (544 silver sentences so far)
  Processed 400/601 abstracts... (791 silver sentences so far)
  Processed 500/601 abstracts... (970 silver sentences so far)
  Processed 600/601 abstracts... (1222 silver sentences so far)

Silver label generation complete!
Total silver sentences: 1225


# Gold+Silver Labels

In [ ]:
# Combine gold (original) + silver labels
combined_sentences = train_sentences + all_silver_sentences
combined_labels = train_labels + all_silver_labels

print(f"Gold sentences:     {len(train_sentences)}")
print(f"Silver sentences:   {len(all_silver_sentences)}")
print(f"Combined total:     {len(combined_sentences)}")

# Tokenize combined dataset
print("\nTokenizing combined dataset...")
combined_encodings = tokenize_and_align_labels(
    combined_sentences, combined_labels, tokenizer
)
combined_dataset = NERDataset(combined_encodings)
combined_loader = DataLoader(
    combined_dataset, batch_size=BATCH_SIZE, shuffle=True
)
print(f"Combined dataset ready: {len(combined_dataset)} samples")

# Retrain with more epochs
EPOCHS_2 = 10
optimizer2 = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps2 = len(combined_loader) * EPOCHS_2
scheduler2 = get_linear_schedule_with_warmup(
    optimizer2,
    num_warmup_steps=100,
    num_training_steps=total_steps2
)

print(f"\nRetraining with {EPOCHS_2} epochs on combined dataset...")
print("=" * 60)

best_dev_f1_2 = 0
best_epoch_2 = 0

for epoch in range(EPOCHS_2):
    model.train()
    total_loss = 0

    for batch in combined_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer2.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer2.step()
        scheduler2.step()

    avg_loss = total_loss / len(combined_loader)
    dev_f1, dev_precision, dev_recall = evaluate(
        model, dev_loader, device
    )

    print(f"Epoch {epoch+1}/{EPOCHS_2}")
    print(f"  Train Loss:    {avg_loss:.4f}")
    print(f"  Dev F1:        {dev_f1:.4f}")
    print(f"  Dev Precision: {dev_precision:.4f}")
    print(f"  Dev Recall:    {dev_recall:.4f}")

    if dev_f1 > best_dev_f1_2:
        best_dev_f1_2 = dev_f1
        best_epoch_2 = epoch + 1
        torch.save(model.state_dict(), "best_scibert_ner_v2.pt")
        print(f"  ✓ Best model saved!")
    print("-" * 40)

print(f"\nRetraining complete!")
print(f"Best Dev F1: {best_dev_f1_2:.4f} at epoch {best_epoch_2}")

Gold sentences:     210
Silver sentences:   1225
Combined total:     1435

Tokenizing combined dataset...
Combined dataset ready: 1435 samples

Retraining with 10 epochs on combined dataset...
Epoch 1/10
  Train Loss:    0.2639
  Dev F1:        0.2724
  Dev Precision: 0.2913
  Dev Recall:    0.2557
  ✓ Best model saved!
----------------------------------------
Epoch 2/10
  Train Loss:    0.2067
  Dev F1:        0.2482
  Dev Precision: 0.3212
  Dev Recall:    0.2023
----------------------------------------
Epoch 3/10
  Train Loss:    0.1665
  Dev F1:        0.2879
  Dev Precision: 0.2821
  Dev Recall:    0.2939
  ✓ Best model saved!
----------------------------------------
Epoch 4/10
  Train Loss:    0.1407
  Dev F1:        0.2799
  Dev Precision: 0.2705
  Dev Recall:    0.2901
----------------------------------------
Epoch 5/10
  Train Loss:    0.1186
  Dev F1:        0.2972
  Dev Precision: 0.2862
  Dev Recall:    0.3092
  ✓ Best model saved!
----------------------------------------
E

# Load best retrained model

In [ ]:
# Load best retrained model
model.load_state_dict(torch.load("best_scibert_ner_v2.pt"))

test_f1, test_precision, test_recall = evaluate(model, test_loader, device)

print("=" * 60)
print("FINAL TEST SET RESULTS - Retrained Model (v2)")
print("=" * 60)
print(f"Test F1:        {test_f1:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall:    {test_recall:.4f}")
print("=" * 60)

# Per entity breakdown
from seqeval.metrics import classification_report

model.eval()
all_preds = []
all_labels_list = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs.logits, dim=2)

        for pred_seq, label_seq in zip(predictions, labels):
            pred_list = []
            label_list = []
            for pred, label in zip(pred_seq, label_seq):
                if label.item() != -100:
                    pred_list.append(id2label[pred.item()])
                    label_list.append(id2label[label.item()])
            all_preds.append(pred_list)
            all_labels_list.append(label_list)

print("\nPer Entity Type Results:")
print(classification_report(all_labels_list, all_preds))

# Comparison summary
print("\n" + "=" * 60)
print("MODEL COMPARISON SUMMARY")
print("=" * 60)
print(f"Model v1 (210 gold sentences,  5 epochs): F1 = 0.2208")
print(f"Model v2 (1236 gold+silver, 10 epochs):   F1 = {test_f1:.4f}")
print(f"Improvement: {((test_f1 - 0.2208) / 0.2208 * 100):.1f}%")

FINAL TEST SET RESULTS - Retrained Model (v2)
Test F1:        0.3382
Test Precision: 0.3710
Test Recall:    0.3108

Per Entity Type Results:
                  precision    recall  f1-score   support

  Climate_Driver       0.24      0.27      0.25        33
Climate_Variable       0.19      0.11      0.14        45
       Ecosystem       0.38      0.39      0.38        38
       Env_Event       0.38      0.49      0.42        35
    Geo_Location       0.47      0.52      0.49        42
  Human_Activity       0.58      0.39      0.46        57
          Policy       0.08      0.03      0.04        38
         Species       1.00      0.12      0.22         8

       micro avg       0.37      0.31      0.34       296
       macro avg       0.41      0.29      0.30       296
    weighted avg       0.36      0.31      0.32       296


MODEL COMPARISON SUMMARY
Model v1 (210 gold sentences,  5 epochs): F1 = 0.2208
Model v2 (1236 gold+silver, 10 epochs):   F1 = 0.3382
Improvement: 53.2%


# Saving the model

In [ ]:

model.save_pretrained("/content/scibert_ner_climate_v2")
tokenizer.save_pretrained("/content/scibert_ner_climate_v2")

# Download to your computer
import shutil
shutil.make_archive("scibert_ner_climate_v2", "zip",
                    "/content/scibert_ner_climate_v2")

print("Model saved and zipped!")
print("Download scibert_ner_climate_v2.zip from the Files panel")
print("\nSummary for milestone report:")
print(f"  Overall F1:        0.3243")
print(f"  Best entity F1:    Geo_Location (0.53)")
print(f"  Worst entity F1:   Policy (0.03)")
print(f"  Training data:     1236 sentences (210 gold + 1026 silver)")
print(f"  Model:             SciBERT fine-tuned, 10 epochs")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved and zipped!
Download scibert_ner_climate_v2.zip from the Files panel

Summary for milestone report:
  Overall F1:        0.3243
  Best entity F1:    Geo_Location (0.53)
  Worst entity F1:   Policy (0.03)
  Training data:     1236 sentences (210 gold + 1026 silver)
  Model:             SciBERT fine-tuned, 10 epochs


# ROUND 2 NER

In [ ]:
# Reset model to best v2 weights
model.load_state_dict(torch.load("best_scibert_ner_v2.pt"))

# New hyperparameters
EPOCHS_3 = 15
BATCH_SIZE_3 = 32
LEARNING_RATE_3 = 1e-5
WARMUP_STEPS_3 = 200

# Retokenize with longer max length
print("Retokenizing with max_length=256...")
train_encodings_v3 = tokenize_and_align_labels(
    combined_sentences, combined_labels, tokenizer, max_length=256
)
dev_encodings_v3 = tokenize_and_align_labels(
    dev_sentences, dev_labels, tokenizer, max_length=256
)
test_encodings_v3 = tokenize_and_align_labels(
    test_sentences, test_labels, tokenizer, max_length=256
)

train_dataset_v3 = NERDataset(train_encodings_v3)
dev_dataset_v3 = NERDataset(dev_encodings_v3)
test_dataset_v3 = NERDataset(test_encodings_v3)

train_loader_v3 = DataLoader(
    train_dataset_v3, batch_size=BATCH_SIZE_3, shuffle=True
)
dev_loader_v3 = DataLoader(
    dev_dataset_v3, batch_size=BATCH_SIZE_3, shuffle=False
)
test_loader_v3 = DataLoader(
    test_dataset_v3, batch_size=BATCH_SIZE_3, shuffle=False
)

optimizer3 = AdamW(
    model.parameters(),
    lr=LEARNING_RATE_3,
    weight_decay=0.01
)
total_steps3 = len(train_loader_v3) * EPOCHS_3
scheduler3 = get_linear_schedule_with_warmup(
    optimizer3,
    num_warmup_steps=WARMUP_STEPS_3,
    num_training_steps=total_steps3
)

print(f"Training configuration v3:")
print(f"  Epochs:        {EPOCHS_3}")
print(f"  Batch size:    {BATCH_SIZE_3}")
print(f"  Learning rate: {LEARNING_RATE_3}")
print(f"  Max length:    256")
print(f"  Warmup steps:  {WARMUP_STEPS_3}")
print(f"  Train batches: {len(train_loader_v3)}")
print("=" * 60)

best_dev_f1_3 = 0
best_epoch_3 = 0

for epoch in range(EPOCHS_3):
    model.train()
    total_loss = 0

    for batch in train_loader_v3:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer3.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer3.step()
        scheduler3.step()

    avg_loss = total_loss / len(train_loader_v3)
    dev_f1, dev_precision, dev_recall = evaluate(
        model, dev_loader_v3, device
    )

    print(f"Epoch {epoch+1}/{EPOCHS_3}")
    print(f"  Train Loss:    {avg_loss:.4f}")
    print(f"  Dev F1:        {dev_f1:.4f}")
    print(f"  Dev Precision: {dev_precision:.4f}")
    print(f"  Dev Recall:    {dev_recall:.4f}")

    if dev_f1 > best_dev_f1_3:
        best_dev_f1_3 = dev_f1
        best_epoch_3 = epoch + 1
        torch.save(model.state_dict(), "best_scibert_ner_v3.pt")
        print(f"  ✓ Best model saved!")
    print("-" * 40)

print(f"\nTraining v3 complete!")
print(f"Best Dev F1: {best_dev_f1_3:.4f} at epoch {best_epoch_3}")

Retokenizing with max_length=256...
Training configuration v3:
  Epochs:        15
  Batch size:    32
  Learning rate: 1e-05
  Max length:    256
  Warmup steps:  200
  Train batches: 45
Epoch 1/15
  Train Loss:    0.1910
  Dev F1:        0.2844
  Dev Precision: 0.2876
  Dev Recall:    0.2813
  ✓ Best model saved!
----------------------------------------
Epoch 2/15
  Train Loss:    0.1831
  Dev F1:        0.2764
  Dev Precision: 0.2828
  Dev Recall:    0.2703
----------------------------------------
Epoch 3/15
  Train Loss:    0.1770
  Dev F1:        0.2831
  Dev Precision: 0.2897
  Dev Recall:    0.2769
----------------------------------------
Epoch 4/15
  Train Loss:    0.1648
  Dev F1:        0.3022
  Dev Precision: 0.3056
  Dev Recall:    0.2989
  ✓ Best model saved!
----------------------------------------
Epoch 5/15
  Train Loss:    0.1576
  Dev F1:        0.2794
  Dev Precision: 0.2970
  Dev Recall:    0.2637
----------------------------------------
Epoch 6/15
  Train Loss:    